In [1]:
from Data_query.trino_config import *
import numpy as np
from visualisation import *
import pytz

In [8]:
stop_trino()

Trino service stopping triggered.


In [9]:
big_workers = 2
workers = 0
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
sleep(40)

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [4]:
iceberg_sql(f"""select * from meta_up23c where site_id = 1954814618 """)

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,...,voltage_class,m_id,avg_pf,std_pf,pf_99,pf_01,n_long,n_lat,distance_km,s_99


In [ ]:
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
sleep(60)
def run_func(args):
    year, month, split_cons = args
    df = iceberg_sql(f"""
                    with data as 
                        (select 
                            site_id, t_stamp,  sum(power*circuit_polarity)/1000 as P_kw
                        from ts join (select site_id, circuit_id, circuit_polarity  from meta_up23c where is_pv=True and {split_cons})
                            as m on ts.circuit_id = m.circuit_id
                        where year = {year} and  month = {month}  and is_pv=True and voltage >= 200 and voltage <= 300 and {split_cons}
                        group by site_id, t_stamp
                            ),
                        daily_site_maxP AS (
                            SELECT 
                                site_id,
                                date_trunc('day', t_stamp + interval '10' hour) AS day,
                                max(P_kw) AS max_P_kw
                            FROM data
                            group by site_id, date_trunc('day', t_stamp + interval '10' hour)
                        ),
                        base AS (
                                SELECT
                                    d.*,
                                    lag(P_kw) OVER (
                                        PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                        ORDER BY t_stamp
                                    ) AS prev_P_kw,
                                    lag(P_kw, 2) OVER (
                                        PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                        ORDER BY t_stamp
                                    ) AS prev_P_kw2,
                                    lag(P_kw, 3) OVER (
                                        PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                        ORDER BY t_stamp
                                    ) AS prev_P_kw3,
                                    lag(P_kw, 4) OVER (
                                        PARTITION BY site_id, date_trunc('day', t_stamp + interval '10' hour)
                                        ORDER BY t_stamp
                                    ) AS prev_P_kw4
                                FROM data d
                            )
                            SELECT 
                                b.site_id
                            from base b 
                                join daily_site_maxP as p on 
                                    b.site_id = p.site_id and date_trunc('day', b.t_stamp + interval '10' hour) = p.day
                                join (select distinct site_id, ac_capacity_kw  from meta_up23c where is_pv=True and {split_cons}) m
                                    on b.site_id = m.site_id
                            where 
                                P_kw >= .2*ac_capacity_kw
                                AND P_kw <= .75*max_P_kw
                                AND ac_capacity_kw >= 5
                                AND prev_P_kw IS NOT NULL
                                AND prev_P_kw2 IS NOT NULL
                                AND prev_P_kw3 IS NOT NULL
                                AND prev_P_kw4 IS NOT NULL
                            group by b.site_id,  date_trunc('day', t_stamp + interval '10' hour) 
                            having min(greatest(P_kw , prev_P_kw , prev_P_kw2, prev_P_kw3, prev_P_kw4) - least(P_kw , prev_P_kw , prev_P_kw2, prev_P_kw3, prev_P_kw4)) < .004
    """)
    flex_export_sites = df['site_id'].unique()
    print(f"Found {len(flex_export_sites)} flex export sites for year={year}, month={month}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    if len(flex_export_sites) == 0:
        # print('hi')
        return None

    # sleep(random.randint(15, 30))  # add some randomness to avoid overwhelming trino with simultaneous queries           
    print(f"Completed year={year}, month={month}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    # print(df.head(1))
    return df
tasks = [(year, month, split_cons) for year in (2025, 2024) for month in range(1, 13)  
     
          for split_cons in ['system.bucket(postcode, 16) <= 3', '(system.bucket(postcode, 16) > 3 and system.bucket(postcode, 16) <= 7)', 
                            '(system.bucket(postcode, 16) > 7 and system.bucket(postcode, 16) <= 11)', 'system.bucket(postcode, 16) > 11'] ]

        #  for split_cons in [f'system.bucket(postcode, 16) = {i}' for i in range(0, 16)]]
         
            
try:         
    df = trino_parallel_batch(run_func, tasks, num_workers=num_workers, batch_size=num_workers)
except Exception as e:
    print(f"Error during data retrieval: {e}")
finally:
    stop_trino()
    pass

Trino service is already running.
Found 25 flex export sites for year=2025, month=1, (bucket > 3 and bucket <= 7)
Completed year=2025, month=1, (bucket > 3 and bucket <= 7)
Found 24 flex export sites for year=2025, month=1, bucket <= 3
Completed year=2025, month=1, bucket <= 3
Found 32 flex export sites for year=2025, month=1, bucket > 11
Completed year=2025, month=1, bucket > 11
Found 35 flex export sites for year=2025, month=1, (bucket > 7 and bucket <= 11)
Completed year=2025, month=1, (bucket > 7 and bucket <= 11)
Found 21 flex export sites for year=2025, month=2, (bucket > 3 and bucket <= 7)
Completed year=2025, month=2, (bucket > 3 and bucket <= 7)
Found 29 flex export sites for year=2025, month=2, bucket <= 3
Completed year=2025, month=2, bucket <= 3
Found 22 flex export sites for year=2025, month=2, bucket > 11
Completed year=2025, month=2, bucket > 11
Found 29 flex export sites for year=2025, month=2, (bucket > 7 and bucket <= 11)
Completed year=2025, month=2, (bucket > 7 and 

In [11]:
ensure_trino_running(worker_desired_count = 1, big_worker_desired_count=0)
sleep(40)

Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [18]:
df['site_id'].nunique()

539

In [14]:
iceberg_exec("""
    ALTER TABLE meta_up23c
    DROP COLUMN  IF EXISTS flex_export_detected 
    """)

Executed


In [15]:
iceberg_exec("""
    ALTER TABLE meta_up23c
    ADD COLUMN  flex_export_detected boolean
    """)

Executed


In [16]:
iceberg_exec(f"""
    UPDATE meta_up23c
        SET flex_export_detected =
            CASE 
                WHEN site_id IN ({','.join(map(str, df['site_id'].unique()))}) THEN TRUE
                ELSE FALSE
            END
    """)

Executed


In [17]:
iceberg_sql('select * from meta_up23c where flex_export_detected = true')

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,...,m_id,avg_pf,std_pf,pf_99,pf_01,n_long,n_lat,distance_km,s_99,flex_export_detected
0,590144646,QLD,4065,153.00,-27.450,Energex,6.62,5.0,NaN,2020-03-12,...,M39,0.999974,8.935812e-05,1.000000,0.999627,153.00,-27.44,1.110000,None,True
1,590144646,QLD,4065,153.00,-27.450,Energex,6.62,5.0,NaN,2020-03-12,...,M39,0.999974,8.935812e-05,1.000000,0.999627,153.00,-27.44,1.110000,None,True
2,590144646,QLD,4065,153.00,-27.450,Energex,6.62,5.0,NaN,2020-03-12,...,M39,0.999974,8.935812e-05,1.000000,0.999627,153.00,-27.44,1.110000,None,True
3,590144646,QLD,4065,153.00,-27.450,Energex,6.62,5.0,NaN,2020-03-12,...,M39,0.999974,8.935812e-05,1.000000,0.999627,153.00,-27.44,1.110000,None,True
4,1923598658,QLD,4814,146.75,-19.305,Ergon,10.00,10.0,NaN,2024-10-29,...,M13,0.994694,3.778494e-03,0.998505,0.980673,146.74,-19.30,1.241018,None,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1678,949148159,VIC,3140,145.35,-37.755,Ausnet,4.00,5.0,NaN,2025-02-28,...,M13,1.000000,1.124995e-07,1.000000,0.999999,145.34,-37.76,1.241018,None,True
1679,949148159,VIC,3140,145.35,-37.755,Ausnet,4.00,5.0,NaN,2025-02-28,...,M13,1.000000,1.124995e-07,1.000000,0.999999,145.34,-37.76,1.241018,None,True
1680,1880837655,SA,5127,138.70,-34.785,SAPN,9.92,8.2,5.0,2019-08-31,...,M13,0.999987,3.986793e-06,1.000000,0.999980,138.70,-34.78,0.555000,None,True
1681,1880837655,SA,5127,138.70,-34.785,SAPN,9.92,8.2,5.0,2019-08-31,...,M13,0.999987,3.986793e-06,1.000000,0.999980,138.70,-34.78,0.555000,None,True
